In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement batch normalization forward pass for 2D input tensors. Given an input tensor of shape [N, C] where N is the batch size and C is the number of features, compute the normalized output using learnable scale (<code>gamma</code>) and shift (<code>beta</code>) parameters.
</p>

<p>
  For each feature channel j, batch normalization computes:
  $$
  \begin{align}
  \mu_j &= \frac{1}{N} \sum_{i=1}^{N} x_{i,j} \\
  \sigma_j^2 &= \frac{1}{N} \sum_{i=1}^{N} (x_{i,j} - \mu_j)^2 \\
  \hat{x}_{i,j} &= \frac{x_{i,j} - \mu_j}{\sqrt{\sigma_j^2 + \epsilon}} \\
  y_{i,j} &= \gamma_j \hat{x}_{i,j} + \beta_j
  \end{align}
  $$
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> tensor</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:  input = [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]  (N=3, C=2)
        gamma = [1.0, 1.0]
        beta = [0.0, 0.0]
        eps = 1e-5
Output: output = [[-1.224, -1.224], [0.0, 0.0], [1.224, 1.224]]
</pre>

<h2>Example 2:</h2>
<pre>
Input:  input = [[0.0, 1.0], [2.0, 3.0]]  (N=2, C=2)
        gamma = [2.0, 0.5]
        beta = [1.0, -1.0]
        eps = 1e-5
Output: output = [[-1.0, -1.5], [3.0, -0.5]]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>N</code> ≤ 10,000</li>
  <li>1 ≤ <code>C</code> ≤ 1,024</li>
  <li><code>eps</code> = 1e-5</li>
  <li>-100.0 ≤ input values ≤ 100.0</li>
  <li>0.1 ≤ gamma values ≤ 10.0</li>
  <li>-10.0 ≤ beta values ≤ 10.0</li>

  <li>Performance is measured with <code>N</code> = 5,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// input, gamma, beta, output are device pointers
extern "C" void solve(const float* input, const float* gamma, const float* beta, float* output,
                      int N, int C, float eps) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# input, gamma, beta, output are tensors on the GPU
@cute.jit
def solve(
    input: cute.Tensor,
    gamma: cute.Tensor,
    beta: cute.Tensor,
    output: cute.Tensor,
    N: cute.Int32,
    C: cute.Int32,
    eps: cute.Float32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input, gamma, beta are tensors on the GPU
@jax.jit
def solve(
    input: jax.Array, gamma: jax.Array, beta: jax.Array, N: int, C: int, eps: float
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# input, gamma, beta, output are device pointers
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    gamma: UnsafePointer[Float32, MutExternalOrigin],
    beta: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
    C: Int32,
    eps: Float32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# input, gamma, beta, output are tensors on the GPU
def solve(
    input: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    output: torch.Tensor,
    N: int,
    C: int,
    eps: float,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# input, gamma, beta, output are tensors on the GPU
def solve(
    input: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    output: torch.Tensor,
    N: int,
    C: int,
    eps: float,
):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/40_batch_normalization/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
